In [1]:
import duckdb
import pandas as pd
from pathlib import Path

DATA_DIR = Path("..") / "data"
PARQUET = (DATA_DIR / "online_retail_II_raw.parquet").as_posix()

print(Path.cwd())

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE TABLE raw AS SELECT * FROM read_parquet('{PARQUET}')")
con.sql("SELECT COUNT(*) AS n_rows FROM raw").show()

C:\Users\kanak\retail-customer-analytics\notebooks
┌─────────┐
│ n_rows  │
│  int64  │
├─────────┤
│ 1067371 │
└─────────┘



In [2]:
con.execute("""
CREATE OR REPLACE TABLE labelled AS
SELECT
    row_number() OVER () AS row_id,
    *,
    CASE
        WHEN source_sheet = 'Year 2010-2011'
             AND InvoiceDate < TIMESTAMP '2010-12-10'
            THEN 'R02_overlap'
        WHEN upper(StockCode) IN ('POST','DOT','M','C2','C3','D','S','BANK CHARGES',
                                  'ADJUST','ADJUST2','AMAZONFEE','CRUK','B','GIFT',
                                  'TEST001','TEST002')
             OR starts_with(StockCode, 'gift_0001_')
            THEN 'R07_non_product'
        WHEN Price <= 0
            THEN 'R06_zero_price'
    END AS exclude_rule
FROM raw
""")
print("labelled")

labelled


In [3]:
con.execute("""
CREATE OR REPLACE TABLE candidates AS
SELECT *,
    row_number() OVER (
        PARTITION BY Invoice, StockCode, Description, Quantity,
                     InvoiceDate, Price, "Customer ID", Country
        ORDER BY row_id
    ) AS copy_number
FROM labelled
WHERE exclude_rule IS NULL
""")
print("candidates")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

candidates


In [4]:
con.execute("""
CREATE OR REPLACE TABLE clean_sales AS
SELECT
    row_id,
    Invoice                           AS invoice,
    StockCode                         AS stock_code,
    Description                       AS description,
    Quantity                          AS quantity,
    InvoiceDate                       AS invoice_ts,
    Price                             AS unit_price,
    TRY_CAST("Customer ID" AS BIGINT) AS customer_id,
    Country                           AS country,
    Quantity * Price                  AS line_value,
    starts_with(Invoice, 'C')         AS is_return
FROM candidates
WHERE copy_number = 1
""")

con.execute("""
CREATE OR REPLACE TABLE excluded_rows AS
SELECT row_id, Invoice, StockCode, Description, Quantity, InvoiceDate, Price,
       "Customer ID", Country, source_sheet, exclude_rule AS rule_id
FROM labelled WHERE exclude_rule IS NOT NULL
UNION ALL
SELECT row_id, Invoice, StockCode, Description, Quantity, InvoiceDate, Price,
       "Customer ID", Country, source_sheet, 'R03_repeat_line' AS rule_id
FROM candidates WHERE copy_number > 1
""")
print("built clean_sales and excluded_rows")

built clean_sales and excluded_rows


In [5]:
con.sql("""
SELECT rule_id, COUNT(*) AS n_rows, ROUND(SUM(Quantity * Price), 2) AS net_value
FROM excluded_rows
GROUP BY rule_id
ORDER BY rule_id
""").show()

con.sql("""
SELECT COUNT(*) AS n_rows, ROUND(SUM(line_value), 2) AS net_value FROM clean_sales
""").show()

n_raw   = con.execute("SELECT COUNT(*) FROM raw").fetchall()[0][0]
n_clean = con.execute("SELECT COUNT(*) FROM clean_sales").fetchall()[0][0]
n_excl  = con.execute("SELECT COUNT(*) FROM excluded_rows").fetchall()[0][0]
print(n_raw, n_clean, n_excl, "adds up:", n_raw == n_clean + n_excl)

┌─────────────────┬────────┬───────────┐
│     rule_id     │ n_rows │ net_value │
│     varchar     │ int64  │  double   │
├─────────────────┼────────┼───────────┤
│ R02_overlap     │  22523 │ 377488.45 │
│ R03_repeat_line │  11790 │  53862.45 │
│ R06_zero_price  │   5973 │       0.0 │
│ R07_non_product │   5813 │  -71499.4 │
└─────────────────┴────────┴───────────┘

┌─────────┬─────────────┐
│ n_rows  │  net_value  │
│  int64  │   double    │
├─────────┼─────────────┤
│ 1021272 │ 18927399.07 │
└─────────┴─────────────┘

1067371 1021272 46099 adds up: True


In [6]:
con.sql("""
SELECT
    COUNT(*) FILTER (WHERE quantity < 0 AND NOT is_return) AS neg_qty_not_return,
    COUNT(*) FILTER (WHERE quantity > 0 AND is_return)     AS pos_qty_return,
    COUNT(*) FILTER (WHERE unit_price <= 0)                AS bad_price,
    COUNT(*) FILTER (WHERE description IS NULL)            AS no_description,
    COUNT(*) FILTER (WHERE customer_id IS NULL)            AS no_customer,
    COUNT(DISTINCT customer_id)                            AS customers,
    MIN(invoice_ts) AS first_ts,
    MAX(invoice_ts) AS last_ts
FROM clean_sales
""").show()

┌────────────────────┬────────────────┬───────────┬────────────────┬─────────────┬───────────┬─────────────────────┬─────────────────────┐
│ neg_qty_not_return │ pos_qty_return │ bad_price │ no_description │ no_customer │ customers │      first_ts       │       last_ts       │
│       int64        │     int64      │   int64   │     int64      │    int64    │   int64   │    timestamp_ns     │    timestamp_ns     │
├────────────────────┼────────────────┼───────────┼────────────────┼─────────────┼───────────┼─────────────────────┼─────────────────────┤
│                  0 │              0 │         0 │              0 │      227089 │      5875 │ 2009-12-01 07:45:00 │ 2011-12-09 12:50:00 │
└────────────────────┴────────────────┴───────────┴────────────────┴─────────────┴───────────┴─────────────────────┴─────────────────────┘



In [7]:
con.sql("""
SELECT
    customer_id IS NOT NULL AS has_customer,
    COUNT(*) AS n_lines,
    ROUND(SUM(line_value) FILTER (WHERE NOT is_return), 2) AS gross_sales,
    ROUND(SUM(line_value) FILTER (WHERE is_return), 2)     AS returns_value,
    ROUND(100 * SUM(line_value) FILTER (WHERE NOT is_return)
          / SUM(SUM(line_value) FILTER (WHERE NOT is_return)) OVER (), 2) AS pct_of_gross_sales
FROM clean_sales
GROUP BY 1
ORDER BY 1
""").show()

┌──────────────┬─────────┬─────────────┬───────────────┬────────────────────┐
│ has_customer │ n_lines │ gross_sales │ returns_value │ pct_of_gross_sales │
│   boolean    │  int64  │   double    │    double     │       double       │
├──────────────┼─────────┼─────────────┼───────────────┼────────────────────┤
│ false        │  227089 │   2575278.9 │      -6472.54 │              13.11 │
│ true         │  794183 │ 17068582.74 │    -709990.03 │              86.89 │
└──────────────┴─────────┴─────────────┴───────────────┴────────────────────┘



In [8]:
print(con.execute('SELECT COUNT(DISTINCT "Customer ID") FROM raw').fetchall()[0][0],
      "distinct customers in raw")
print(con.execute("SELECT COUNT(DISTINCT customer_id) FROM clean_sales").fetchall()[0][0],
      "distinct customers in clean_sales")

con.sql("""
SELECT COUNT(*) AS customers_with_no_purchase
FROM (
    SELECT customer_id
    FROM clean_sales
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
    HAVING COUNT(*) FILTER (WHERE NOT is_return) = 0
)
""").show()

5942 distinct customers in raw
5875 distinct customers in clean_sales
┌────────────────────────────┐
│ customers_with_no_purchase │
│           int64            │
├────────────────────────────┤
│                         23 │
└────────────────────────────┘



In [9]:
con.sql("""
SELECT COUNT(DISTINCT "Customer ID") AS customers_only_in_excluded_rows
FROM excluded_rows
WHERE "Customer ID" IS NOT NULL
  AND "Customer ID" NOT IN (SELECT customer_id FROM clean_sales WHERE customer_id IS NOT NULL)
""").show()

┌─────────────────────────────────┐
│ customers_only_in_excluded_rows │
│              int64              │
├─────────────────────────────────┤
│                              67 │
└─────────────────────────────────┘



In [10]:
CLEAN_PQ = (DATA_DIR / "clean_sales.parquet").as_posix()
EXCL_PQ  = (DATA_DIR / "excluded_rows.parquet").as_posix()

con.execute(f"COPY clean_sales TO '{CLEAN_PQ}' (FORMAT PARQUET)")
con.execute(f"COPY excluded_rows TO '{EXCL_PQ}' (FORMAT PARQUET)")
print("saved")

saved
